# MINE: Regime I, Evo 2 (Local-Global Decoupling)

Separate notebook for Evo 2 evaluation. Requires flash-attn + evo2 package.
Saves results to `./results/mine/` for combined analysis.

In [ ]:
import os
import numpy as np

PHASE = 'full'  # 'quick' or 'full'

OUTPUT_BASE = './results/mine_v2_three_regimes/'
RESULTS_DIR = OUTPUT_BASE + 'results'
CACHE_DIR   = OUTPUT_BASE + 'cache'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

CONFIG = {
    'quick': {
        'n_sequences': 100,
        'seq_lengths': [100, 200],
        'mine_epochs': 200,
        'mine_batch_size': 64,
        'mine_lr': 1e-4,
        'mine_hidden': [256, 128],
        'n_runs': 3,
    },
    'full': {
        'n_sequences': 200,
        'seq_lengths': [100, 200, 400],
        'mine_epochs': 500,
        'mine_batch_size': 64,
        'mine_lr': 1e-4,
        'mine_hidden': [256, 128],
        'n_runs': 5,
    },
}

RANDOM_SEED = 320
config = CONFIG[PHASE]

print(f"Phase: {PHASE.upper()}")
print(f"Sequences per class: {config['n_sequences']}")
print(f"Sequence lengths: {config['seq_lengths']}")
print(f"MINE epochs: {config['mine_epochs']}")
print(f"MINE independent runs: {config['n_runs']}")
print(f"Results: {RESULTS_DIR}")

# Evo 2 specific config
EVO2_CHECKPOINT = "evo2_7b"
EVO2_EMBEDDING_LAYER = "blocks.28.mlp.l3"
EVO2_CONTEXT_LENGTH = 8192
EVO2_LOCAL_WINDOW = 128  # bp window for local MI estimation

## Install Dependencies

This is the heavy install: flash-attn builds from source (~10-15 min on A100).

In [ ]:
import os, sys

print('Installing core dependencies...')
!pip install -q matplotlib seaborn pandas scipy scikit-learn numpy
!pip install -q ninja

# Pin torch to 2.7.1+cu128 (Arc Institute recommended for evo2/flash-attn)
print('Pinning torch to 2.7.1+cu128...')
!pip install -q torch==2.7.1 --index-url https://download.pytorch.org/whl/cu128

import torch
print(f'torch {torch.__version__} | CUDA {torch.version.cuda}')

# Build flash-attn from source (~10-15 min on A100, one-time cost)
print('Building flash-attn 2.8.0.post2 from source (~10-15 min)...')
!pip install flash-attn==2.8.0.post2 --no-build-isolation

!pip install -q evo2

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

from evo2 import Evo2
print('\nEvo2 package loaded successfully')
print('Done!')

## MINE Core Implementation

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class MINEStatisticsNetwork(nn.Module):
    # Statistics network T_theta for MINE.
    # Takes concatenated (x, x_hat) as input and outputs a scalar.
    def __init__(self, input_dim, hidden_dims=[256, 128]):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            layers.extend([nn.Linear(prev_dim, h), nn.ReLU(), nn.Dropout(0.1)])
            prev_dim = h
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def mine_estimate(T_net, joint_batch, marginal_batch):
    # Compute the Donsker-Varadhan lower bound on MI.
    # I(X;X_hat) >= E_joint[T(x,x_hat)] - log(E_marginal[exp(T(x,x_hat))])
    t_joint = T_net(joint_batch)
    t_marginal = T_net(marginal_batch)
    first_term = t_joint.mean()
    second_term = torch.logsumexp(t_marginal, dim=0) - np.log(t_marginal.shape[0])
    mi_lb = first_term - second_term
    return mi_lb


def run_mine(X_bio, X_emb, n_epochs, batch_size, lr, hidden_dims, seed,
             device='cuda', verbose=True):
    # Run a single MINE estimation.
    torch.manual_seed(seed)
    np.random.seed(seed)

    N = X_bio.shape[0]
    d_bio = X_bio.shape[1]
    d_emb = X_emb.shape[1]
    input_dim = d_bio + d_emb

    X_bio_t = torch.FloatTensor(
        (X_bio - X_bio.mean(0)) / (X_bio.std(0) + 1e-8)
    ).to(device)
    X_emb_t = torch.FloatTensor(
        (X_emb - X_emb.mean(0)) / (X_emb.std(0) + 1e-8)
    ).to(device)

    T_net = MINEStatisticsNetwork(input_dim, hidden_dims).to(device)
    optimizer = optim.Adam(T_net.parameters(), lr=lr)

    mi_estimates = []

    for epoch in range(n_epochs):
        perm = torch.randperm(N)
        marginal_perm = torch.randperm(N)
        epoch_mi = []

        for i in range(0, N - batch_size + 1, batch_size):
            idx = perm[i:i + batch_size]
            marginal_idx = marginal_perm[i:i + batch_size]

            x_bio_batch = X_bio_t[idx]
            x_emb_batch = X_emb_t[idx]

            joint = torch.cat([x_bio_batch, x_emb_batch], dim=1)
            x_emb_marginal = X_emb_t[marginal_idx]
            marginal = torch.cat([x_bio_batch, x_emb_marginal], dim=1)

            mi_lb = mine_estimate(T_net, joint, marginal)
            loss = -mi_lb
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(T_net.parameters(), max_norm=5.0)
            optimizer.step()

            epoch_mi.append(mi_lb.item())

        mean_mi = np.mean(epoch_mi)
        mi_estimates.append(mean_mi)

        if verbose and (epoch + 1) % 50 == 0:
            print(f"  Epoch {epoch+1}/{n_epochs}: MI_lb = {mean_mi:.4f}")

    tail = max(1, len(mi_estimates) // 10)
    final_mi = np.mean(mi_estimates[-tail:])
    return mi_estimates, final_mi


def run_mine_with_ci(X_bio, X_emb, config, device='cuda', label=''):
    # Run MINE multiple times and compute confidence intervals.
    all_traces = []
    all_finals = []

    for run_idx in range(config['n_runs']):
        seed = RANDOM_SEED + run_idx * 100
        if label:
            print(f"\n  [{label}] Run {run_idx+1}/{config['n_runs']} (seed={seed})")
        traces, final_mi = run_mine(
            X_bio, X_emb,
            n_epochs=config['mine_epochs'],
            batch_size=config['mine_batch_size'],
            lr=config['mine_lr'],
            hidden_dims=config['mine_hidden'],
            seed=seed,
            device=device,
            verbose=(run_idx == 0),
        )
        all_traces.append(traces)
        all_finals.append(final_mi)
        print(f"    Final MI = {final_mi:.4f}")

    mean_mi = np.mean(all_finals)
    std_mi = np.std(all_finals)
    print(f"\n  [{label}] MI = {mean_mi:.4f} +/- {std_mi:.4f}")
    return all_traces, all_finals, mean_mi, std_mi


print("MINE implementation ready")
print(f"  Architecture: input -> {config['mine_hidden']} -> 1")
print(f"  Training: {config['mine_epochs']} epochs, lr={config['mine_lr']}")

## MINE Sanity Check

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print('='*70)
print('MINE SANITY CHECK: Correlated Gaussians with known MI')
print('='*70)

sanity_results = []
for rho in [0.0, 0.3, 0.6, 0.9]:
    N_sanity = 2000
    rng_sc = np.random.default_rng(RANDOM_SEED)
    z1 = rng_sc.standard_normal(N_sanity)
    z2 = rng_sc.standard_normal(N_sanity)
    x_sanity = z1.reshape(-1, 1).astype(np.float32)
    y_sanity = (rho * z1 + np.sqrt(1 - rho**2) * z2).reshape(-1, 1).astype(np.float32)

    mi_true = -0.5 * np.log(1 - rho**2) if rho != 0 else 0.0

    traces_sc, mi_est = run_mine(
        x_sanity, y_sanity, n_epochs=300, batch_size=64,
        lr=1e-4, hidden_dims=[128, 64], seed=RANDOM_SEED,
        device=device, verbose=False,
    )
    sanity_results.append({'rho': rho, 'mi_true': mi_true, 'mi_est': mi_est})
    tol_ok = abs(mi_est - mi_true) < max(0.15, 0.3 * mi_true)
    status = 'PASS' if tol_ok else 'FAIL'
    print(f'  rho={rho:.1f}: MI_true={mi_true:.4f}, MI_est={mi_est:.4f}  [{status}]')

print('\nIf estimates track the true values, MINE is calibrated.')
print('Proceed to real model evaluation.\n')

## DNA Ground Truth

In [ ]:
import urllib.request

DNA_BASES = ['A', 'C', 'G', 'T']

def compute_dna_features(sequences):
    '''Compute GC content + dinucleotide frequencies.'''
    dinucs = [a+b for a in DNA_BASES for b in DNA_BASES]  # 16
    features = []
    for seq in sequences:
        L = len(seq)
        gc = (seq.count('G') + seq.count('C')) / L
        di_freq = np.array([seq.count(d) / max(1, L-1) for d in dinucs])
        features.append(np.concatenate([[gc], di_freq]))
    return np.array(features, dtype=np.float32)


def compute_dna_features_local(sequences, start, end):
    '''Compute DNA features for a specific window [start, end) across all sequences.'''
    dinucs = [a+b for a in DNA_BASES for b in DNA_BASES]
    features = []
    for seq in sequences:
        window = seq[start:end]
        L = len(window)
        if L < 2:
            features.append(np.zeros(17, dtype=np.float32))
            continue
        gc = (window.count('G') + window.count('C')) / L
        di_freq = np.array([window.count(d) / max(1, L-1) for d in dinucs])
        features.append(np.concatenate([[gc], di_freq]).astype(np.float32))
    return np.array(features, dtype=np.float32)


def cleanup_gpu():
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def fetch_real_genomic_regions(n_regions, region_length, seed=320):
    '''Fetch real human genomic regions from UCSC API.'''
    cache_file = f'{CACHE_DIR}/real_genomic_regions_{n_regions}_{region_length}.txt'
    if os.path.exists(cache_file):
        with open(cache_file) as f:
            seqs = [line.strip() for line in f if line.strip()]
        print(f'  Loaded {len(seqs)} cached genomic regions')
        return seqs

    print(f'  Fetching {n_regions} real genomic regions ({region_length:,} bp each)...')
    rng = np.random.default_rng(seed)

    chroms = [
        ('chr1', 248_956_422), ('chr2', 242_193_529), ('chr3', 198_295_559),
        ('chr4', 190_214_555), ('chr5', 181_538_259), ('chr6', 170_805_979),
        ('chr7', 159_345_973), ('chr8', 145_138_636), ('chr9', 138_394_717),
        ('chr10', 133_797_422), ('chr11', 135_086_622), ('chr12', 133_275_309),
        ('chr13', 114_364_328), ('chr14', 107_043_718), ('chr15', 101_991_189),
        ('chr16', 90_338_345), ('chr17', 83_257_441), ('chr18', 80_373_285),
        ('chr19', 58_617_616), ('chr20', 64_444_167), ('chr21', 46_709_983),
        ('chr22', 50_818_468),
    ]

    sequences = []
    attempts = 0
    max_attempts = n_regions * 5

    while len(sequences) < n_regions and attempts < max_attempts:
        attempts += 1
        chrom_name, chrom_len = chroms[rng.integers(len(chroms))]
        margin = int(chrom_len * 0.1)
        start = rng.integers(margin, chrom_len - margin - region_length)
        end = start + region_length

        try:
            import json as _json
            url = f'https://api.genome.ucsc.edu/getData/sequence?genome=hg38;chrom={chrom_name};start={start};end={end}'
            req = urllib.request.Request(url, headers={'User-Agent': 'Python/mine_v2'})
            with urllib.request.urlopen(req, timeout=15) as resp:
                data = _json.loads(resp.read().decode())
                seq = data.get('dna', '').upper()

            if len(seq) >= region_length:
                seq = seq[:region_length]
                n_valid = sum(1 for b in seq if b in 'ACGT')
                if n_valid / len(seq) > 0.95:
                    seq = ''.join(b if b in 'ACGT' else rng.choice(DNA_BASES)
                                   for b in seq)
                    sequences.append(seq)
                    if len(sequences) % 20 == 0:
                        print(f'    Fetched {len(sequences)}/{n_regions}', flush=True)
        except Exception as e:
            if attempts % 50 == 0:
                print(f'    Retry {attempts}: {e}')
            continue

    if len(sequences) < n_regions:
        print(f'  WARNING: Only got {len(sequences)}/{n_regions} from UCSC, padding with random DNA.')
        while len(sequences) < n_regions:
            sequences.append(''.join(rng.choice(DNA_BASES, size=region_length)))

    with open(cache_file, 'w') as f:
        f.write('\n'.join(sequences))
    print(f'  Cached {len(sequences)} genomic regions')

    gc_vals = [(s.count('G') + s.count('C')) / len(s) for s in sequences]
    print(f'  GC content: mean={np.mean(gc_vals):.3f}, std={np.std(gc_vals):.3f}, '
          f'range=[{np.min(gc_vals):.3f}, {np.max(gc_vals):.3f}]')
    return sequences


# Fetch DNA for Evo 2
n_dna = config['n_sequences'] * 2
print('Fetching real genomic DNA for Evo 2...')
dna_sequences = fetch_real_genomic_regions(n_dna, EVO2_CONTEXT_LENGTH, seed=RANDOM_SEED)
X_bio_dna_global = compute_dna_features(dna_sequences)
print(f"DNA ground truth (global): {X_bio_dna_global.shape}")

## PCA + MI Ceiling

In [ ]:
from sklearn.decomposition import PCA

MINE_PCA_DIM = 50

def pca_reduce_embeddings(X_emb, n_components=MINE_PCA_DIM, seed=RANDOM_SEED):
    '''Reduce embedding dimensionality via PCA for stable MINE estimation.'''
    if X_emb.shape[1] <= n_components:
        print(f"    PCA: skipped (dim={X_emb.shape[1]} <= {n_components})")
        return X_emb, None

    actual_components = min(n_components, X_emb.shape[0], X_emb.shape[1])
    pca = PCA(n_components=actual_components, random_state=seed)
    X_reduced = pca.fit_transform(X_emb).astype(np.float32)
    var_explained = pca.explained_variance_ratio_.sum()
    print(f"    PCA: {X_emb.shape[1]}d -> {actual_components}d "
          f"({var_explained:.1%} variance retained)")
    return X_reduced, pca


def calibrate_mi_ceiling(X_bio, config, device='cuda', label='', noise_scale=0.1):
    '''Estimate MI ceiling by running MINE with X_bio as both X and X_hat.'''
    print(f"\n  Calibrating MI ceiling for {label}...")
    rng = np.random.default_rng(RANDOM_SEED)
    X_perfect = X_bio + noise_scale * rng.standard_normal(X_bio.shape).astype(np.float32)

    traces, finals, mean_mi, std_mi = run_mine_with_ci(
        X_bio, X_perfect, config, device=device,
        label=f'Ceiling-{label}'
    )
    print(f"  MI Ceiling ({label}): {mean_mi:.4f} +/- {std_mi:.4f} nats")
    return mean_mi, std_mi


print(f"PCA reduction: all embeddings reduced to {MINE_PCA_DIM}d before MINE")
print("MI ceiling calibration: run MINE(X_bio, X_bio + noise) for each distribution")

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print('='*70)
print('MI CEILING CALIBRATION: DNA')
print('='*70)

mi_ceiling_dna, mi_ceiling_dna_std = calibrate_mi_ceiling(
    X_bio_dna_global, config, device=device, label='DNA'
)

print(f'\nCeiling: DNA = {mi_ceiling_dna:.4f} +/- {mi_ceiling_dna_std:.4f} nats')

## Random Baseline (DNA)

In [ ]:
import time

print('\n' + '='*70)
print('RANDOM BASELINE: DNA Distribution')
print('='*70)

rng_rand = np.random.default_rng(RANDOM_SEED)

X_emb_rand_dna = rng_rand.standard_normal(
    (X_bio_dna_global.shape[0], MINE_PCA_DIM)
).astype(np.float32)

t0 = time.time()
traces, finals, rand_mi_dna, rand_std_dna = run_mine_with_ci(
    X_bio_dna_global, X_emb_rand_dna, config, device=device,
    label='Random-DNA')

noise_floor_dna = rand_mi_dna + 2 * rand_std_dna
print(f'\nNoise floor (DNA): {noise_floor_dna:.4f} nats')

## Evo 2 Embedding Function

In [ ]:
def make_evo2_embed_fn(checkpoint_name=EVO2_CHECKPOINT, batch_size=1):
    '''Load Evo 2 (7B) via Vortex. Returns two embedding functions:
    - embed_global: mean-pool across full context (for global MI)
    - embed_local: per-position embeddings for windowed local MI
    '''
    from evo2 import Evo2

    print(f'Loading Evo 2 ({checkpoint_name}) via Vortex...')
    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

    evo2_model = Evo2(checkpoint_name)
    n_params = sum(p.numel() for p in evo2_model.model.parameters()) / 1e6
    layer_name = EVO2_EMBEDDING_LAYER

    if torch.cuda.is_available():
        mem = torch.cuda.memory_allocated() / 1e9
        print(f'  Params: {n_params:.1f}M | GPU mem: {mem:.2f} GB')
    print(f'  Embedding layer: {layer_name}')

    @torch.no_grad()
    def embed_global(sequences):
        '''Mean-pool across full context for global MI.'''
        embeddings = []
        for idx, seq in enumerate(sequences):
            if (idx + 1) % 10 == 0:
                print(f'    Global: {idx+1}/{len(sequences)}', end='\r')
            input_ids = torch.tensor(
                evo2_model.tokenizer.tokenize(seq), dtype=torch.int,
            ).unsqueeze(0).to(device)
            _, layer_embs = evo2_model(
                input_ids, return_embeddings=True, layer_names=[layer_name],
            )
            emb = layer_embs[layer_name]
            pooled = emb.mean(dim=1).squeeze(0)
            embeddings.append(pooled.cpu().float().numpy())
        print()
        return np.array(embeddings, dtype=np.float32)

    @torch.no_grad()
    def embed_local(sequences, window_start, window_end):
        '''Extract embeddings for a specific window [start, end) for local MI.'''
        embeddings = []
        for idx, seq in enumerate(sequences):
            if (idx + 1) % 10 == 0:
                print(f'    Local [{window_start}:{window_end}]: {idx+1}/{len(sequences)}', end='\r')
            input_ids = torch.tensor(
                evo2_model.tokenizer.tokenize(seq), dtype=torch.int,
            ).unsqueeze(0).to(device)
            _, layer_embs = evo2_model(
                input_ids, return_embeddings=True, layer_names=[layer_name],
            )
            emb = layer_embs[layer_name]
            w_start = min(window_start, emb.shape[1])
            w_end = min(window_end, emb.shape[1])
            if w_end > w_start:
                local_pooled = emb[0, w_start:w_end, :].mean(dim=0)
            else:
                local_pooled = emb[0, 0, :] * 0
            embeddings.append(local_pooled.cpu().float().numpy())
        print()
        return np.array(embeddings, dtype=np.float32)

    print(f'Evo 2 ready ({n_params:.0f}M params, layer={layer_name})')
    return embed_global, embed_local, evo2_model

## Run MINE: Regime I (Evo 2)

In [ ]:
import time
import pandas as pd

mine_results = []
mine_traces = {}

NOTEBOOK_TAG = 'evo2'

print('\n' + '='*70)
print('REGIME I: Evo 2 (Local-Global Decoupling)')
print('='*70)

embed_global_fn, embed_local_fn, model_evo2 = make_evo2_embed_fn(
    checkpoint_name=EVO2_CHECKPOINT, batch_size=1
)

evo2_dna_seqs = []
for seq in dna_sequences:
    if len(seq) < EVO2_CONTEXT_LENGTH:
        evo2_dna_seqs.append(seq + 'N' * (EVO2_CONTEXT_LENGTH - len(seq)))
    else:
        evo2_dna_seqs.append(seq[:EVO2_CONTEXT_LENGTH])

# ======== GLOBAL MI ========
print('\n--- Evo 2: GLOBAL MI (mean-pooled across full context) ---')
cache_global = f'{CACHE_DIR}/evo2_7b_mine_global_{EVO2_CONTEXT_LENGTH}.npy'
if os.path.exists(cache_global):
    X_emb_global = np.load(cache_global)
    print(f'  Loaded cached global embeddings: {X_emb_global.shape}')
else:
    print(f'  Computing global embeddings ({len(evo2_dna_seqs)} seqs)...')
    X_emb_global = embed_global_fn(evo2_dna_seqs)
    X_emb_global = np.nan_to_num(X_emb_global, nan=0.0, posinf=1e6, neginf=-1e6)
    np.save(cache_global, X_emb_global)
    print(f'  Saved: {X_emb_global.shape}')

X_emb_global_pca, _ = pca_reduce_embeddings(X_emb_global)

t0 = time.time()
traces, finals, mean_mi_global, std_mi_global = run_mine_with_ci(
    X_bio_dna_global, X_emb_global_pca, config, device=device,
    label='Evo2-Global'
)
mine_results.append({
    'model': 'Evo2_Global', 'regime': 'I', 'seq_length': EVO2_CONTEXT_LENGTH,
    'mi_mean': mean_mi_global, 'mi_std': std_mi_global, 'mi_runs': finals,
    'time_min': (time.time()-t0)/60,
    'ground_truth': 'dna', 'mi_ceiling': mi_ceiling_dna,
    'mi_normalized': mean_mi_global / max(mi_ceiling_dna, 1e-8),
    'scope': 'global',
})
mine_traces['Evo2-Global'] = traces

# ======== LOCAL MI (multiple windows) ========
local_windows = [
    (500, 500 + EVO2_LOCAL_WINDOW),
    (2000, 2000 + EVO2_LOCAL_WINDOW),
    (4000, 4000 + EVO2_LOCAL_WINDOW),
]

local_mi_values = []

for w_start, w_end in local_windows:
    print(f'\n--- Evo 2: LOCAL MI [positions {w_start}:{w_end}] ---')
    cache_local = f'{CACHE_DIR}/evo2_7b_mine_local_{w_start}_{w_end}.npy'
    if os.path.exists(cache_local):
        X_emb_local = np.load(cache_local)
        print(f'  Loaded cached local embeddings: {X_emb_local.shape}')
    else:
        print(f'  Computing local embeddings...')
        X_emb_local = embed_local_fn(evo2_dna_seqs, w_start, w_end)
        X_emb_local = np.nan_to_num(X_emb_local, nan=0.0, posinf=1e6, neginf=-1e6)
        np.save(cache_local, X_emb_local)
        print(f'  Saved: {X_emb_local.shape}')

    X_emb_local_pca, _ = pca_reduce_embeddings(X_emb_local)

    X_bio_local = compute_dna_features_local(evo2_dna_seqs, w_start, w_end)

    t0 = time.time()
    traces, finals, mean_mi_local, std_mi_local = run_mine_with_ci(
        X_bio_local, X_emb_local_pca, config, device=device,
        label=f'Evo2-Local-{w_start}'
    )

    local_mi_values.append(mean_mi_local)

    mine_results.append({
        'model': f'Evo2_Local_{w_start}', 'regime': 'I', 'seq_length': EVO2_LOCAL_WINDOW,
        'mi_mean': mean_mi_local, 'mi_std': std_mi_local, 'mi_runs': finals,
        'time_min': (time.time()-t0)/60,
        'ground_truth': 'dna_local', 'mi_ceiling': mi_ceiling_dna,
        'mi_normalized': mean_mi_local / max(mi_ceiling_dna, 1e-8),
        'scope': 'local', 'window': f'{w_start}-{w_end}',
    })
    mine_traces[f'Evo2-Local-{w_start}'] = traces

# Decoupling summary
avg_local_mi = np.mean(local_mi_values)
print(f'\n--- Local-Global Decoupling Summary (Regime I) ---')
print(f'  Global MI:  {mean_mi_global:.4f} +/- {std_mi_global:.4f} nats')
print(f'  Local MI:   {avg_local_mi:.4f} nats (mean across {len(local_windows)} windows)')
if avg_local_mi > 0:
    ratio = mean_mi_global / avg_local_mi
    print(f'  Ratio (global/local): {ratio:.4f}')
    print(f'  Decoupling factor: {1/ratio:.1f}x')

# Add random baseline
mine_results.append({
    'model': 'Random_DNA', 'regime': 'baseline', 'seq_length': 0,
    'mi_mean': rand_mi_dna, 'mi_std': rand_std_dna, 'mi_runs': [],
    'time_min': 0,
    'ground_truth': 'dna', 'mi_ceiling': mi_ceiling_dna,
    'mi_normalized': rand_mi_dna / max(mi_ceiling_dna, 1e-8),
})

del model_evo2
cleanup_gpu()

## Save Results

In [ ]:
import pandas as pd
import json as _json

# Save results to Drive
results_file = f'{RESULTS_DIR}/{NOTEBOOK_TAG}_results_{PHASE}.json'
with open(results_file, 'w') as f:
    # Convert numpy types for JSON serialization
    clean_results = []
    for r in mine_results:
        cr = {}
        for k, v in r.items():
            if isinstance(v, np.floating):
                cr[k] = float(v)
            elif isinstance(v, np.integer):
                cr[k] = int(v)
            elif isinstance(v, np.ndarray):
                cr[k] = v.tolist()
            elif isinstance(v, list):
                cr[k] = [float(x) if isinstance(x, np.floating) else x for x in v]
            else:
                cr[k] = v
        clean_results.append(cr)
    _json.dump(clean_results, f, indent=2)

# Save traces
traces_file = f'{RESULTS_DIR}/{NOTEBOOK_TAG}_traces_{PHASE}.json'
clean_traces = {}
for k, v in mine_traces.items():
    clean_traces[k] = [list(map(float, t)) for t in v]
with open(traces_file, 'w') as f:
    _json.dump(clean_traces, f)

print(f'Saved results: {results_file}')
print(f'Saved traces: {traces_file}')

df = pd.DataFrame(clean_results)
print('\n' + df.to_string(index=False))